In [16]:
# Imports
# Data handling
import pandas as pd
import pickle

# Train-test split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [11]:
# Load cleaned dataset
import pandas as pd

df = pd.read_csv("Big Data SSP/data_cleaned.csv")

df.head()

,category_list,funding_total_usd,country_code,state_code,funding_rounds,target,founded_year,first_funding_year,last_funding_year
0,Media,10000000.0,IND,16,1,1,2010.0,2015.0,2015
1,Application Platforms,700000.0,USA,DE,2,1,2014.0,2014.0,2014
2,Apps,3406878.0,NaN,NaN,1,1,2010.0,2014.0,2014
3,Curated Web,2000000.0,CHN,22,1,1,2007.0,2008.0,2008
4,Software,2000000.0,USA,IL,1,1,2010.0,2014.0,2014


In [17]:
# Split features and target
X = df.drop(columns=['target'])
y = df['target']

print(X.shape)
print(y.shape)

(66368, 8)
(66368,)


In [ ]:
# Numeric features
numerical_features = [
    'funding_total_usd',
    'funding_rounds',
    'founded_year',
    'first_funding_year',
    'last_funding_year'
]

# Categorical features
categorical_features = [
    'category_list',
    'country_code',
    'state_code'
]

In [19]:
# Create train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(53094, 8)
(13274, 8)


In [20]:
# Scale numeric features
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

In [21]:
# Handle missing values and encode categories
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='unknown'
    )),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

In [22]:
# Apply transformations by column type
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

In [23]:
# Transform training data
X_train_processed = preprocessor.fit_transform(X_train)

print(X_train_processed.shape)

(53094, 1141)


In [24]:
# Number of unique values per category
for col in categorical_features:
    print(col, ":", X_train[col].nunique())

category_list : 698
country_code : 135
state_code : 301


# Dummy Classifier for Baseline

- T   90% of companies successful? 10% failure 💡
- S : no 10% failure and not fail is the rest
- T : huge inbabalance in data be careful 
- S : complicated to differenciate? 
- T : prediction must be define  score by comparaison, good to get better explanation of result, try multiclass , not just binary, 1st model binary, 2nd model determine between 3 outputs , check deep learning transformer softmax multiclass

In [25]:
# Majority class baseline
dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyClassifier(strategy='most_frequent'))
])

dummy_pipeline.fit(X_train, y_train)

dummy_pred = dummy_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, dummy_pred))
print("Precision:", precision_score(y_test, dummy_pred))
print("Recall:", recall_score(y_test, dummy_pred))
print("F1 Score:", f1_score(y_test, dummy_pred))

Accuracy: 0.9059816182009944
Precision: 0.9059816182009944
Recall: 1.0
F1 Score: 0.9506719367588933


In [26]:
# because the dataset is 90% successful startups!

In [27]:
# Logistic regression baseline
logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ))
])

logreg_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['funding_total_usd',
                                                   'funding_rounds',
                                                   'founded_year',
                                                   'first_funding_year',
                                                   'last_funding_year']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='unknown',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['category_list',
                                                   'country_code',
                                                   'state_code'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    random_state=42))])

In [28]:
# Predictions
y_pred = logreg_pipeline.predict(X_test)

# Probabilities
y_prob = logreg_pipeline.predict_proba(X_test)[:, 1]

In [29]:
# Model evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7322585505499473
Precision: 0.9529512403763901
Recall: 0.7410610344254116
F1 Score: 0.833754326878099
ROC AUC: 0.7528388678163125


## Lets see what this means!

In [31]:
# When the model predicts a startup will succeed,
# it is correct 95% of the time.

# The model identifies about 74% of successful startups.

# ROC AUC: 75% The model is learning meaningful patterns from the data.

In [33]:
# Confusion matrix
print(confusion_matrix(y_test, y_pred))

# TN  FP
# FN  TP

[[ 808  440]
 [3114 8912]]


In [34]:
dummy_pipeline.fit(X_train, y_train)

dummy_pred = dummy_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, dummy_pred))
print("F1:", f1_score(y_test, dummy_pred))

Accuracy: 0.9059816182009944
F1: 0.9506719367588933


- E : possible to improve on accuracy 💡

In [37]:
# Save model pipeline
with open("startup_success_model.pkl", "wb") as file:
    pickle.dump(logreg_pipeline, file)

print("Model saved")

Model saved


In [38]:
# pkl
import os

os.listdir()

['startup_success_model.pkl',
 'archive.zip',
 '.ipynb_checkpoints',
 'raw_data',
 '.gitignore',
 'Big Data SSP Preprocessing .ipynb',
 '01_Baseline_ModelMV.ipynb',
 'SSP.ipynb',
 'Big Data SSP',
 'Big Data SSP Data Cleaning .ipynb',
 'archive.zip:Zone.Identifier',
 'README.md',
 '.git',
 'archive.zip:mshield']